# Project 02 — Iris Flower Classifier

**Difficulty:** Beginner  
**Skills:** scikit-learn, classification, EDA, model evaluation  
**Dataset:** sklearn built-in Iris (no download needed)

## Objective
Build a multi-class classifier to predict iris species (Setosa, Versicolor, Virginica) from flower measurements. This is the 'Hello World' of machine learning.

## Pipeline
1. Load & explore data
2. Visualise features
3. Split into train/test
4. Train multiple models
5. Evaluate and compare
6. Visualise decision boundary

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 1. Load & Explore

In [ ]:
# Load the dataset
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df.columns = ['sepal_length','sepal_width','petal_length','petal_width']
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

print('Shape:', df.shape)
print('Classes:', df['species'].unique())
print('Class counts:')
print(df['species'].value_counts())
print()
df.head()

In [ ]:
print(df.describe().T.round(3))

## 2. Exploratory Visualisation

In [ ]:
# Pair plot — reveals separability
g = sns.pairplot(df, hue='species', diag_kind='kde',
                 plot_kws={'alpha': 0.7}, height=2.2,
                 palette='Set2')
g.figure.suptitle('Iris Feature Pair Plot', y=1.02, fontsize=13)
plt.show()

In [ ]:
# Box plots — feature distribution by species
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = ['sepal_length','sepal_width','petal_length','petal_width']

for ax, feat in zip(axes.flatten(), features):
    sns.boxplot(data=df, x='species', y=feat,
                palette='Set2', ax=ax)
    ax.set_title(feat.replace('_',' ').title())
    ax.set_xlabel('')

plt.suptitle('Feature Distributions by Species', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[features].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()
print('Key insight: petal features are highly correlated with each other')

## 3. Prepare Data

In [ ]:
X = df[features].values
y = iris.target   # 0=setosa, 1=versicolor, 2=virginica

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print('Class distribution in train:', np.bincount(y_train))
print('Class distribution in test: ', np.bincount(y_test))

## 4. Train and Compare Multiple Models

In [ ]:
# Define models as pipelines (scaler + classifier)
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()),
                                      ('clf', LogisticRegression(max_iter=200, random_state=42))]),
    'K-Nearest Neighbors': Pipeline([('scaler', StandardScaler()),
                                      ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'Decision Tree':       Pipeline([('clf', DecisionTreeClassifier(max_depth=4, random_state=42))]),
    'Random Forest':       Pipeline([('clf', RandomForestClassifier(n_estimators=100, random_state=42))]),
    'SVM (RBF kernel)':    Pipeline([('scaler', StandardScaler()),
                                      ('clf', SVC(kernel='rbf', C=1.0, random_state=42))]),
}

results = []
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results.append({
        'Model': name,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Accuracy': test_acc
    })

results_df = pd.DataFrame(results).sort_values('Test Accuracy', ascending=False)
print(results_df.round(4).to_string(index=False))

In [ ]:
# Visualise model comparison
fig, ax = plt.subplots(figsize=(10, 4))

bars = ax.barh(results_df['Model'], results_df['Test Accuracy'],
               color='steelblue', edgecolor='white', alpha=0.85)
ax.errorbar(results_df['CV Mean'], results_df['Model'],
            xerr=results_df['CV Std'], fmt='ro', capsize=4, label='CV score ± std')
for bar in bars:
    ax.text(bar.get_width()+0.003, bar.get_y()+bar.get_height()/2,
            f'{bar.get_width():.1%}', va='center', fontsize=10)

ax.set_xlim(0.8, 1.05)
ax.set_xlabel('Accuracy')
ax.set_title('Model Comparison — Iris Classification', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Best Model — Detailed Evaluation

In [ ]:
# Use Random Forest as best/most robust model
best_model = models['Random Forest']
y_pred = best_model.predict(X_test)

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=iris.target_names)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix — Random Forest')

# Feature importances
rf = best_model.named_steps['clf']
feat_imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)
feat_imp.plot.barh(color='steelblue', ax=axes[1], edgecolor='white')
for i, v in enumerate(feat_imp):
    axes[1].text(v+0.005, i, f'{v:.3f}', va='center')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

## 6. Visualise Decision Tree

In [ ]:
dt = models['Decision Tree'].named_steps['clf']

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(dt, feature_names=features, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Decision Tree Visualisation (max_depth=4)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Making Predictions on New Data

In [ ]:
new_flowers = pd.DataFrame({
    'sepal_length': [5.1, 6.7, 6.3],
    'sepal_width':  [3.5, 3.0, 2.5],
    'petal_length': [1.4, 5.2, 5.0],
    'petal_width':  [0.2, 2.3, 1.9]
})

predictions = best_model.predict(new_flowers.values)
probabilities = best_model.predict_proba(new_flowers.values)

for i, (pred, probs) in enumerate(zip(predictions, probabilities)):
    species = iris.target_names[pred]
    confidence = probs.max()
    print(f'Flower {i+1}: {species:12s}  (confidence: {confidence:.1%})')

## Key Findings

| Finding | Detail |
|---------|--------|
| **Best accuracy** | All models achieve ≥96% — Iris is a well-separable dataset |
| **Setosa is trivial** | Perfectly linearly separable from the others |
| **Versicolor vs Virginica** | The hard boundary — only petal features separate them well |
| **Most important features** | `petal_width` and `petal_length` (> sepal measurements) |
| **Recommended model** | Random Forest — robust and provides feature importance |

## What to Try Next
- Tune hyperparameters with `GridSearchCV`
- Try with only 2 features to plot a 2D decision boundary
- Add more data augmentation with `SMOTE` (though data is balanced here)
- Graduate to a harder dataset: Breast Cancer, Wine Quality